# 1 · Compare land-cover models

**Question:** does changing the model or augmentation improve the land-cover baseline on the same validation tiles?

Keep imagery, labels, split membership, and preprocessing fixed so the comparison is meaningful. Start with the small model from notebook 0, describe its parts as a model recipe, then add augmentation. Compare validation IoU alongside accuracy and, optionally, the per-class scores from notebook 0; reserve the test set for the final choice. Record the settings and seed with each run. A single short run is a workflow demonstration, not proof that one architecture is better.

**Data and interpretation.** These are expert-annotated training labels from [Tait et al. (2021)](https://doi.org/10.1594/PANGAEA.933475), paired with Sentinel-2 L2A imagery in the [teaching subset](https://huggingface.co/datasets/fatmur/dynamicworld-300). They have ten classes, including cloud, and nodata 255. They are not the automated nine-class Dynamic World probability product.

Keep tiles in their original train/validation/test splits before making chips. Nearby or overlapping tiles can still leak spatial information: inspect footprints before claiming geographic generalization. These short runs are workflow checks, not evidence of mapping accuracy.


**Run:** use the repository `.venv` kernel with `examples/` as the working directory. Requires `uv sync` and a ~400 MB download; CPU training is supported.

## Reuse the same data and splits

This notebook defines its Dataset again so it runs in a fresh kernel. Use the same eight prepared training pairs and two validation pairs as the baseline. Keep the image band order and label IDs unchanged.

In [ ]:
from pathlib import Path

import lightning as L
import torch
from huggingface_hub import snapshot_download
from torch import nn
from torch.utils.data import DataLoader, Dataset

import geosave as gs

# One seed for the whole notebook, so the model's first weights and the shuffled
# batches repeat from run to run.
L.seed_everything(0)

root = Path(
    snapshot_download(
        "fatmur/dynamicworld-300",
        repo_type="dataset",
        revision="5e65026ad47e1f02f9a48a3efee1028db3bb0d1a",
        local_dir="data/dynamic_world",
    )
)

## Read a pair when training asks for it

The Dataset reads the prepared pairs and converts them to tensors. Image values become reflectance; label IDs stay unchanged. Data acquisition and preparation are separate work, introduced in notebook 2.

In [ ]:
class DynamicWorld(Dataset):
    def __init__(self, images, labels):
        self.images = images
        self.labels = labels

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        with gs.open_raster(self.images[index]) as scene:
            image = scene[["B02", "B03", "B04", "B08"]]
            image = image.gs.mask_and_scale()
            image = image.gs.to_tensor(dtype="float32")

        with gs.open_raster(self.labels[index]) as scene:
            label = scene["label"].gs.to_tensor(dtype="int64")

        return image, label

## Keep the training and validation splits separate

Use eight prepared training pairs and two validation pairs. Training shuffles the pairs. Validation keeps their order so we can find the corresponding image and label when reviewing a prediction.

In [ ]:
train_images = sorted((root / "train/images").glob("*.tif"))[:8]
train_labels = sorted((root / "train/labels").glob("*.tif"))[:8]
val_images = sorted((root / "val/images").glob("*.tif"))[:2]
val_labels = sorted((root / "val/labels").glob("*.tif"))[:2]

train = DynamicWorld(train_images, train_labels)
val = DynamicWorld(val_images, val_labels)

train_loader = DataLoader(train, batch_size=8, shuffle=True)
val_loader = DataLoader(val, batch_size=8)

## The same three parts

Ordinary PyTorch modules, plus declarations GeoSave uses to connect and build them.

In [ ]:
from geosave.model import ModelChain, Published, build_model, chain_step
from geosave.model.registry import register_model


@register_model("encoder", "tiny")
class Encoder(nn.Module):
    channels: Published[int]

    def __init__(self, channels: int = 16):
        super().__init__()

        self.channels = channels

        self.net = nn.Sequential(
            nn.Conv2d(4, channels, 3, stride=2, padding=1),
            nn.ReLU(),
        )

    @chain_step()
    def forward(self, image: torch.Tensor) -> torch.Tensor:
        features = self.net(image)  # (B, 4, H, W) -> (B, channels, H/2, W/2)
        return features

The decoder uses the encoder's channel count to restore the spatial size.

In [ ]:
@register_model("decoder", "upsample")
class Decoder(nn.Module):
    feature_channels: Published[int]

    def __init__(self, channels: int):
        super().__init__()

        self.feature_channels = channels

        self.net = nn.Sequential(
            nn.Upsample(scale_factor=2),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.ReLU(),
        )

    @chain_step()
    def forward(self, features: torch.Tensor) -> torch.Tensor:
        feature_map = self.net(features)
        return feature_map

The head turns those features into land-cover class scores.

In [ ]:
@register_model("head", "pixel")
class Head(nn.Module):
    def __init__(self, channels: int):
        super().__init__()

        self.net = nn.Conv2d(channels, 10, 1)

    @chain_step(head=True)
    def forward(self, feature_map: torch.Tensor) -> torch.Tensor:
        return self.net(feature_map)  # (B, channels, H, W) -> (B, 10, H, W)

## Connect parts by what they read and return

`chain_step` declares the named values a part reads and produces. `ModelChain` wires them into a graph: the encoder makes `features`, the decoder `feature_map`, the head the scores. Displaying the model shows that flow.

In [ ]:
model = ModelChain(
    encoder=Encoder(),
    decoder=Decoder(channels=16),
    head=Head(channels=16),
)
model

## Write the model as a recipe

`@register_model` names each class within its stage. Selected by name, the model is plain data, the form a YAML or JSON config holds:

In [ ]:
model = build_model(
    {
        "encoder": {"name": "tiny", "init_args": {"channels": 16}},
        "decoder": {"name": "upsample"},
        "head": {"name": "pixel"},
    }
)
model

GeoSave's `segmentation` head replaces ours: same 1x1 layer, but the class count comes from the label file. We keep width 16, so the flips below are the only change from notebook 0.

In [ ]:
label = gs.open_raster(train_labels[0])["label"]
# 255 marks unlabelled pixels, not a class
num_classes = sum(
    code != 255 for code in label.gs.legend.classes
)  # number of actual classes

model = build_model(
    {
        "encoder": {"name": "tiny", "init_args": {"channels": 16}},
        "decoder": {"name": "upsample"},
        "head": {"name": "segmentation", "init_args": {"num_classes": num_classes}},
    }
)
model

## Augmentation

[Kornia](https://kornia.readthedocs.io) augments whole batches. `AugmentationSequential` applies the same flip to image and labels; `data_keys` says which is which.

In [ ]:
import kornia.augmentation as K
import lightning as L
from torchmetrics import MetricCollection
from torchmetrics.classification import MulticlassAccuracy, MulticlassJaccardIndex


class Segmenter(L.LightningModule):
    def __init__(self, model, num_classes):
        super().__init__()

        self.model = model

        self.augment = K.AugmentationSequential(
            K.RandomHorizontalFlip(),
            K.RandomVerticalFlip(),
            data_keys=["image", "mask"],
            keepdim=True,
        )
        self.val_metrics = MetricCollection(
            {
                "miou": MulticlassJaccardIndex(
                    num_classes=num_classes, ignore_index=255
                ),
                "accuracy": MulticlassAccuracy(
                    num_classes=num_classes, ignore_index=255, average="micro"
                ),
            },
            prefix="val_",
        )

        self.criterion = nn.CrossEntropyLoss(ignore_index=255)

    def forward(self, image):
        return self.model(image=image)

    def training_step(self, batch, batch_index):
        image, label = self.augment(*batch)  # (B, C, H, W), (B, H, W)
        logits = self(image)  # (B, classes, H, W)
        loss = self.criterion(logits, label)

        self.log("train_loss", loss)

        return loss

    def validation_step(self, batch, batch_index):
        image, label = batch  # (B, C, H, W), (B, H, W)
        logits = self(image)  # (B, classes, H, W)
        loss = self.criterion(logits, label)

        self.log("val_loss", loss)
        self.val_metrics(logits, label)
        self.log_dict(self.val_metrics)

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=1e-2)


module = Segmenter(model, num_classes)

In [ ]:
import warnings

from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger

# The tiles are small and few; Lightning's hint to add loader workers is noise here.
warnings.filterwarnings("ignore", ".*does not have many workers.*")

# Each notebook logs to a folder of its own; version 0 is its one run.
logger = TensorBoardLogger("lightning_logs", name="01_model", version=0)
# Save the last epoch for an explicit future resume.
checkpoint = ModelCheckpoint(save_last=True)

trainer = L.Trainer(
    max_epochs=6,
    log_every_n_steps=1,
    logger=logger,
    callbacks=[checkpoint],
)
trainer.fit(module, train_loader, val_loader)

## Compare the runs

TensorBoard draws `00_basics` and `01_model` on the same axes. Six short epochs are noisy, so read the gap loosely.

### Optional: inspect training logs

Run this in a separate cell when you want TensorBoard. It starts a local service.

```python
%load_ext tensorboard
%tensorboard --logdir lightning_logs
```

## Predict from an image

Use the image's grid and the saved class definitions to create the prediction map. No reference label is needed.

In [ ]:
import xarray as xr
import yaml

classes = yaml.safe_load((root / "classes.yaml").read_text())
image = gs.open_raster(val_images[0])

reflectance = image[["B02", "B03", "B04", "B08"]].gs.mask_and_scale()
pixels = reflectance.gs.to_tensor(dtype="float32")
batch = pixels.unsqueeze(0).to(module.device)

module.eval()
with torch.inference_mode():
    logits = module(batch)
    prediction = logits.argmax(dim=1)[0].cpu().numpy()

predicted = xr.DataArray(prediction, dims=("y", "x"), name="label")
predicted = predicted.gs.write_grid(image.gs.geobox)
predicted = predicted.astype("uint8")
predicted = predicted.gs.write_nodata(classes["ignore_index"])
predicted = predicted.gs.write_legend(classes["class_map"], classes["color_map"])

image.gs.plot(xlabel="") + predicted.gs.plot(xlabel="")

### Compare with a reference when one is available

This validation image happens to have a reference label. Open it separately to inspect the model's errors.

In [ ]:
reference = gs.open_raster(val_labels[0])["label"]
reference.gs.plot(title="Reference") + predicted.gs.plot(title="Prediction")

## What happened

We described the baseline's parts as a reusable model recipe and trained with paired flips. Compare the validation scores and maps with notebook 0 before deciding whether the augmentation helped. The next lesson builds our own image/label pairs.

## More things you can do with the experiment (optional)

The model experiment is complete. You can continue to [notebook 2](02_sources.ipynb). The alternatives below are ideas for later experiments, not prerequisites.

### Separate architecture gains from augmentation gains
This run changes both how the model is assembled and its training augmentation. For a controlled follow-up, keep the architecture fixed and train once with flips and once without, using distinct logger versions. Compare the same validation maps and class metrics.

### Choose the next experiment from visible errors
If errors cluster around boundaries, compare the same crop with a different decoder. If a class is consistently absent, inspect its training support before increasing model size. A short run suggests an experiment; it does not establish which architecture is best.

### Take the project beyond the teaching images
A better model still needs the right inputs at a new location. [Notebook 2](02_sources.ipynb) starts with the existing labels and builds aligned imagery, then acquires observations over the forest we want to map.

### Try a different way to restore image detail

This decoder resizes to the image's size, so it reads `features` and `image`:

If the maps look too coarse around boundaries, try bilinear resizing and train it as a separate experiment. Building a candidate below does not train it or replace the model already fitted above.

In [ ]:
class ResizeDecoder(nn.Module):
    @chain_step()
    def forward(self, features: torch.Tensor, image: torch.Tensor) -> torch.Tensor:
        feature_map = nn.functional.interpolate(
            features, size=image.shape[-2:], mode="bilinear", align_corners=False
        )
        return feature_map


candidate = ModelChain(
    encoder=Encoder(),
    decoder=ResizeDecoder(),
    head=Head(channels=16),
)
candidate

The encoder and head are unchanged: each step receives only the values it asks for, and gradients still flow back to the encoder.

### Test whether a wider model helps

`channels: Published[int]` on the encoder passes its width to parts built after it. `build_model` constructs the parts in order, so one change widens all three:

In [ ]:
candidate = build_model(
    {"encoder": Encoder(channels=32), "decoder": Decoder, "head": Head}
)
candidate

### Find a pretrained model for the next experiment

Pretrained encoders, decoders and heads are registered the same way. `list_models` shows how each is called, what it publishes and what it needs:

In [ ]:
from geosave.model import list_models

list_models()

Naming a part lists what can run before and after it:

In [ ]:
list_models("upsample")